In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl qwen-vl-utils
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q "torchao==0.16.0"

In [ ]:
from fastapi import FastAPI, UploadFile, Form
import uvicorn
import threading
from PIL import Image
import io
import math
from peft import PeftModel
import os
import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

In [ ]:
SYSTEM_PROMPT = """### ROLE
You are a specialized Landmark Recognition Expert. Your ONLY task is to identify landmarks from images.

### STRICT OUTPUT RULE (CRITICAL)
Your response MUST ALWAYS start with the exact phrase: "Based on the visual features, this is ".
DO NOT use any other introduction. DO NOT explain. DO NOT add conversational filler.

- Format: Based on the visual features, this is [Exact Name of Landmark], [City/Location]
- If you cannot identify it: Outside scope
- If it is a generic object: Outside scope

### GUIDELINES
1. FAMOUS STREETS/SQUARES: Treat famous urban areas (e.g., Ba Dinh Square, Nguyen Hue Walking Street) as landmarks.
2. SPECIFICITY: If a landmark is inside a square, name the square if the image shows the broader area.
3. OUTSIDE SCOPE: Respond "Outside scope" for coding, math, general chat, or non-landmark images.
FEW-SHOT EXAMPLES:
User: (Image of a street in Saigon)
Assistant: Based on the visual features, this is Pham Ngu Lao Street, Ho Chi Minh City

User: (Image of a cat)
Assistant: Outside scope

User: (Text) Write a Python script for sorting.
Assistant: Outside scope
"""

In [ ]:
class LandmarkModel:
    def __init__(self, model, processor, device="cuda"):
        self.model = model
        self.processor = processor
        self.device = device

        if hasattr(self.processor, "tokenizer"):
            self.processor.tokenizer.padding_side = "left"
        else:
            self.processor.padding_side = "left"

        self.model.eval()

    def pad_to_square(self, img):
        w, h = img.size
        size = max(w, h)
        new_img = Image.new("RGB", (size, size), (0, 0, 0))
        new_img.paste(img, ((size - w) // 2, (size - h) // 2))
        return new_img

    def smart_resize_by_tokens(self, img, target_tokens=196, patch_size = 28):
        w, h = img.size
        target_grid = int(math.sqrt(target_tokens))

        scale = (target_grid * patch_size) / max(w, h)
        scale = min(scale, 1.0)

        new_w = int(w * scale)
        new_h = int(h * scale)

        new_w = max(patch_size, (new_w // patch_size) * patch_size)
        new_h = max(patch_size, (new_h // patch_size) * patch_size)

        try:
            resample_filter = Image.Resampling.LANCZOS
        except AttributeError:
            resample_filter = Image.LANCZOS

        return img.resize((new_w, new_h), resample_filter)

    def preprocess_image(self, image_path):

        if isinstance(image_path, str):
            image = Image.open(image_path).convert("RGB")
        else:
            image = image_path.convert("RGB")

        image = self.pad_to_square(image)
        image = self.smart_resize_by_tokens(image)
        return image

    def predict(self, prompt, image_path=None, system_prompt=SYSTEM_PROMPT):
        content = []

        if image_path:
            processed_image = self.preprocess_image(image_path)
            content.append({"type": "image", "image": processed_image})

        content.append({"type": "text", "text": prompt})

        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": content}
        ]

        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        image_inputs = None
        video_inputs = None
        if image_path:
            from qwen_vl_utils import process_vision_info
            image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.device)

        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return output_text


In [ ]:
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
hf_id = "imbee510/qwen2-5-vl-landmark-vietnam-lora"

ft_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

ft_model = PeftModel.from_pretrained(ft_model, hf_id)

In [ ]:
app = FastAPI()

vision_model = LandmarkModel(ft_model, processor)
@app.post("/predict_landmark")
async def detect(file: UploadFile, prompt: str = Form(...)):
    # Đọc ảnh từ request
    image_data = await file.read()
    image = Image.open(io.BytesIO(image_data))

    result_text = vision_model.predict(prompt=prompt, image_path=image)

    return {
        "landmark": result_text,
        "confidence": 0 # Mock Data
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

threading.Thread(target=run_server, daemon=True).start()

In [ ]:
# TODO: Chạy lệnh dưới để tunnel ra bên ngoài CHÚ Ý: NHẤN PHẢI RỒI CHỌN COPY TRONG MENU, NẾU BẤM CTRL + C SẼ TẮT KẾT NỐI
!ssh -T -o StrictHostKeyChecking=no -p 443 -R0:localhost:8001 a.pinggy.io